# SAC Training for Irrigation Control

Trains a Soft Actor-Critic agent on the 130-agent crop-soil ABM.

**Kaggle constraints:**
- 30 GPU-hours/week
- 9-hour session limit
- Checkpoints every 10k steps for crash recovery

**Strategy:** Train on dry/100% first (hardest scenario), then transfer to others.
5 seeds × 500k steps each. Estimated time: ~2-3h per seed on T4 GPU.

In [ ]:
# Install dependencies
!pip install -q stable-baselines3[extra] gymnasium tensorboard

In [ ]:
# Clone the thesis repository
import os
REPO_DIR = '/kaggle/working/thesis'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/taratorbati/thesis.git {REPO_DIR}
os.chdir(REPO_DIR)

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
SCENARIO = 'dry'       # Start with hardest scenario
BUDGET_PCT = 100
TOTAL_TIMESTEPS = 500_000
SEEDS = [0, 1, 2, 3, 4]

# Kaggle output
OUTPUT_DIR = '/kaggle/working/rl_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Quick sanity check: create env and step once
from src.rl.gym_env import IrrigationEnv

env = IrrigationEnv(scenario=SCENARIO, budget_pct=BUDGET_PCT)
obs, info = env.reset()
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env.action_space}')

action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)
print(f'Step 0: reward={reward:.4f}, yield={info["yield_kg_ha"]:.0f} kg/ha')
del env

In [ ]:
# ── Training loop over seeds ─────────────────────────────────────
from src.rl.train import train_sac
from pathlib import Path
import time

SESSION_START = time.time()
SESSION_LIMIT_S = 8.5 * 3600  # 8.5h safety margin within 9h limit

for seed in SEEDS:
    elapsed = time.time() - SESSION_START
    if elapsed > SESSION_LIMIT_S:
        print(f'\nSession time limit approaching ({elapsed/3600:.1f}h). Stopping.')
        print(f'Resume from the last checkpoint in the next session.')
        break
    
    run_name = f'sac_{SCENARIO}_{BUDGET_PCT}pct_seed{seed}'
    final_model = Path(OUTPUT_DIR) / run_name / f'{run_name}_final.zip'
    
    if final_model.exists():
        print(f'Seed {seed}: already complete, skipping.')
        continue
    
    # Check for checkpoint to resume from
    checkpoint_dir = Path(OUTPUT_DIR) / run_name / 'checkpoints'
    resume_path = None
    if checkpoint_dir.exists():
        checkpoints = sorted(checkpoint_dir.glob('*.zip'))
        if checkpoints:
            resume_path = str(checkpoints[-1])
            print(f'Seed {seed}: resuming from {resume_path}')
    
    print(f'\n{"="*60}')
    print(f'Training seed {seed} ({elapsed/3600:.1f}h elapsed)')
    print(f'{"="*60}\n')
    
    train_sac(
        scenario=SCENARIO,
        budget_pct=BUDGET_PCT,
        total_timesteps=TOTAL_TIMESTEPS,
        seed=seed,
        output_dir=OUTPUT_DIR,
        dem_path=str(Path(REPO_DIR) / 'gilan_farm.tif'),
        checkpoint_freq=10_000,
        eval_freq=10_000,
        resume_path=resume_path,
        verbose=1,
    )

In [ ]:
# ── Copy results to Kaggle output for download ───────────────────
import shutil
output_kaggle = Path('/kaggle/working/output')
output_kaggle.mkdir(exist_ok=True)

for seed in SEEDS:
    run_name = f'sac_{SCENARIO}_{BUDGET_PCT}pct_seed{seed}'
    run_dir = Path(OUTPUT_DIR) / run_name
    
    if run_dir.exists():
        # Copy best model and final model
        for src in run_dir.glob('**/*.zip'):
            dst = output_kaggle / src.name
            shutil.copy2(src, dst)
        
        # Copy logs and configs
        for src in run_dir.glob('*.json'):
            dst = output_kaggle / src.name
            shutil.copy2(src, dst)

print(f'Output files: {list(output_kaggle.iterdir())}')

In [ ]:
# ── Quick evaluation of best model ───────────────────────────────
from src.rl.gym_env import IrrigationEnv
from stable_baselines3 import SAC
import numpy as np

# Load best model from seed 0
best_model_path = Path(OUTPUT_DIR) / f'sac_{SCENARIO}_{BUDGET_PCT}pct_seed0' / 'best_model' / 'best_model.zip'
if best_model_path.exists():
    model = SAC.load(str(best_model_path), device='cpu')
    
    env = IrrigationEnv(scenario=SCENARIO, budget_pct=BUDGET_PCT)
    obs, _ = env.reset()
    
    total_reward = 0
    for day in range(env.season_days):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    
    print(f'Evaluation results:')
    print(f'  Yield: {info["yield_kg_ha"]:.0f} kg/ha')
    print(f'  Water used: {info["water_used_mm"]:.1f} mm')
    print(f'  Budget remaining: {info["budget_remaining"]:.1f} mm')
    print(f'  Total reward: {total_reward:.4f}')
else:
    print('No best model found yet.')